<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio 1, Modulo 2 Tema 4</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>


## Leccion 5: Ejercicio propuesto y solucion

Este cuaderno desarrolla el ejercicio de optimizacion y monitorizacion de una aplicacion Spark batch utilizando como origen de datos el fichero reducido `sales.csv`.

El objetivo no es solo obtener un resultado correcto, sino observar como influyen el particionado, el cache, el shuffle y Adaptive Query Execution en el rendimiento de una aplicacion Spark.

## Dataset utilizado

El fichero de entrada es:

- `sales.csv`, ubicado en la misma carpeta que este cuaderno.
- Tamano aproximado: 20 MB.
- Cada fila representa la actividad diaria de un producto en una tienda.

Columnas principales:

- `product_id`
- `store_id`
- `date`
- `sales`
- `revenue`
- `stock`
- `price`
- `promo_type_1`
- `promo_type_2`

## Recomendacion de uso

Ejecuta el cuaderno bloque a bloque y revisa siempre:

1. El esquema del DataFrame.
2. El numero de filas procesadas.
3. El numero de particiones.
4. El tiempo de ejecucion de cada experimento.
5. Las evidencias de Spark UI cuando ejecutes el notebook en local.

## 1. Enunciado del ejercicio

Implementa, ejecuta y analiza una aplicacion Spark batch sobre `sales.csv`. La solucion debe realizar las siguientes tareas:

1. Leer el dataset CSV con cabecera.
2. Validar y convertir los tipos de datos principales.
3. Descartar registros con ventas o ingresos negativos.
4. Crear columnas derivadas: ingreso medio por unidad, indicador de rotura de stock y ano-mes.
5. Calcular ventas e ingresos por tienda, producto, mes y tipo de promocion.
6. Obtener rankings de tiendas y productos por ingresos.
7. Guardar agregados en formato Parquet.
8. Comparar configuraciones de Spark relacionadas con particiones, cache y AQE.
9. Documentar metricas y conclusiones.

### Metricas que se deben recoger

- Tiempo total de ejecucion.
- Numero de filas procesadas.
- Numero de particiones usadas.
- Configuracion de `spark.sql.shuffle.partitions`.
- Estado de `spark.sql.adaptive.enabled`.
- Observaciones sobre Spark UI: jobs, stages, tasks, shuffle y spills si aparecen.

## 2. Preparacion del entorno

Si PySpark no esta instalado en el entorno, ejecuta la siguiente celda. Si ya estas trabajando en un entorno Spark configurado, puedes omitirla.

In [ ]:
# Ejecutar solo si PySpark no esta instalado.
# %pip install pyspark==3.5.1

## 3. Creacion de la sesion Spark

La sesion se crea con un nombre explicito para identificar la ejecucion en Spark UI. En modo notebook, parametros como `spark.driver.memory` o el numero de cores locales deben definirse antes de crear la sesion.

In [ ]:
from pathlib import Path
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("M2T4_Ejercicio_Sales_Optimizacion")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
print("Spark version:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

## 4. Definicion de rutas

El cuaderno esta pensado para ejecutarse desde `modulo2/Tema4`, donde esta ubicado `sales.csv`. Si lo ejecutas desde otra ubicacion, ajusta `sales_path`.

In [ ]:
sales_path = Path("sales.csv")
output_dir = Path("output") / "metricas_sales"

if not sales_path.exists():
    raise FileNotFoundError(f"No se encuentra el fichero de entrada: {sales_path.resolve()}")

sales_size_mb = sales_path.stat().st_size / 1_000_000
print(f"Fichero de entrada: {sales_path.resolve()}")
print(f"Tamano aproximado: {sales_size_mb:.2f} MB")

## 5. Carga inicial del CSV

Se lee el CSV con cabecera. En una solucion productiva seria recomendable definir un esquema explicito desde el inicio; aqui se muestra primero la carga basica para inspeccionar el dataset.

In [ ]:
raw_sales_df = spark.read.csv(str(sales_path), header=True, inferSchema=True)

raw_sales_df.printSchema()
raw_sales_df.show(5, truncate=False)

## 6. Solucion: limpieza, tipado y columnas derivadas

La solucion prepara el DataFrame para analisis batch. Se convierten columnas numericas, se normaliza la fecha y se crean variables que luego se usaran en agregaciones.

In [ ]:
sales_df = (
    raw_sales_df
    .withColumn("date", F.to_date("date"))
    .withColumn("sales", F.col("sales").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("stock", F.col("stock").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
    .filter(F.col("sales").isNotNull())
    .filter(F.col("revenue").isNotNull())
    .filter(F.col("sales") >= 0)
    .filter(F.col("revenue") >= 0)
    .withColumn(
        "avg_revenue_per_unit",
        F.when(F.col("sales") > 0, F.col("revenue") / F.col("sales")).otherwise(F.lit(0.0))
    )
    .withColumn("is_stockout", F.when(F.col("stock") <= 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("has_promo_1", F.when(F.col("promo_type_1").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("has_promo_2", F.when(F.col("promo_type_2").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
)

sales_df.printSchema()
sales_df.show(5, truncate=False)

## 7. Metricas basicas del dataset

Estas metricas sirven para documentar el volumen procesado y para comparar posteriores experimentos.

In [ ]:
start = time.perf_counter()

row_count = sales_df.count()
partition_count = sales_df.rdd.getNumPartitions()

elapsed = time.perf_counter() - start

print(f"Filas procesadas: {row_count:,}".replace(",", "."))
print(f"Particiones del DataFrame: {partition_count}")
print(f"Tiempo de conteo: {elapsed:.2f} segundos")

## 8. Agregaciones principales

Las siguientes operaciones generan shuffle porque agrupan por claves distribuidas. Son buenas candidatas para observar stages, tareas y volumen de shuffle en Spark UI.

In [ ]:
kpi_store_month = (
    sales_df
    .groupBy("store_id", "year_month")
    .agg(
        F.sum("sales").alias("total_units"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("avg_revenue_per_unit").alias("avg_revenue_per_unit"),
        F.avg("is_stockout").alias("stockout_ratio"),
        F.count("*").alias("rows")
    )
)

kpi_store_month.orderBy(F.desc("total_revenue")).show(10, truncate=False)

In [ ]:
kpi_product = (
    sales_df
    .groupBy("product_id")
    .agg(
        F.sum("sales").alias("total_units"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("price").alias("avg_price"),
        F.avg("is_stockout").alias("stockout_ratio")
    )
)

kpi_product.orderBy(F.desc("total_revenue")).show(10, truncate=False)

In [ ]:
kpi_promo = (
    sales_df
    .groupBy("promo_type_1", "promo_type_2")
    .agg(
        F.count("*").alias("rows"),
        F.sum("sales").alias("total_units"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("avg_revenue_per_unit").alias("avg_revenue_per_unit")
    )
)

kpi_promo.orderBy(F.desc("total_revenue")).show(10, truncate=False)

## 9. Escritura de resultados en Parquet

La escritura permite comprobar el numero y tamano de ficheros generados. En Spark UI tambien permite observar stages de escritura.

In [ ]:
start = time.perf_counter()

(
    kpi_store_month
    .write
    .mode("overwrite")
    .parquet(str(output_dir / "kpi_store_month"))
)

(
    kpi_product
    .write
    .mode("overwrite")
    .parquet(str(output_dir / "kpi_product"))
)

elapsed = time.perf_counter() - start
print(f"Resultados escritos en: {output_dir.resolve()}")
print(f"Tiempo de escritura: {elapsed:.2f} segundos")

## 10. Funcion auxiliar para comparar experimentos

La funcion ejecuta el mismo trabajo con distintas configuraciones. Asi se pueden comparar tiempos de forma reproducible dentro del notebook.

In [ ]:
def run_sales_job(df, shuffle_partitions=32, adaptive_enabled=True, use_cache=False):
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions))
    spark.conf.set("spark.sql.adaptive.enabled", str(adaptive_enabled).lower())

    work_df = df
    if use_cache:
        work_df = df.cache()
        work_df.count()

    start = time.perf_counter()

    result_df = (
        work_df
        .groupBy("store_id", "year_month", "promo_type_1")
        .agg(
            F.sum("sales").alias("total_units"),
            F.sum("revenue").alias("total_revenue"),
            F.avg("stock").alias("avg_stock"),
            F.avg("is_stockout").alias("stockout_ratio")
        )
    )

    result_count = result_df.count()
    elapsed = time.perf_counter() - start

    if use_cache:
        work_df.unpersist()

    return {
        "shuffle_partitions": shuffle_partitions,
        "adaptive_enabled": adaptive_enabled,
        "use_cache": use_cache,
        "result_rows": result_count,
        "elapsed_seconds": round(elapsed, 3),
        "input_partitions": df.rdd.getNumPartitions(),
    }

## 11. Experimentos de configuracion

En un entorno local pequeno, los resultados pueden variar entre ejecuciones. Lo importante es interpretar la tendencia y justificarla con metricas y observaciones de Spark UI.

In [ ]:
experiments = [
    {"shuffle_partitions": 8, "adaptive_enabled": False, "use_cache": False},
    {"shuffle_partitions": 32, "adaptive_enabled": False, "use_cache": False},
    {"shuffle_partitions": 200, "adaptive_enabled": False, "use_cache": False},
    {"shuffle_partitions": 32, "adaptive_enabled": True, "use_cache": False},
    {"shuffle_partitions": 32, "adaptive_enabled": True, "use_cache": True},
]

experiment_results = [run_sales_job(sales_df, **config) for config in experiments]
experiment_results_df = spark.createDataFrame(experiment_results)

experiment_results_df.orderBy("elapsed_seconds").show(truncate=False)

## 12. Interpretacion de resultados

Puntos que se deben analizar en el informe:

1. Si `spark.sql.shuffle.partitions=200` genera demasiadas tareas pequenas para un fichero de 20 MB.
2. Si `spark.sql.shuffle.partitions=8` reduce overhead o limita paralelismo.
3. Si AQE mejora el tiempo al coalescer particiones tras el shuffle.
4. Si `cache()` aporta mejora cuando el DataFrame se reutiliza o si solo consume memoria adicional.
5. Si el numero de cores locales esta equilibrado con el tamano del dataset.

En Spark UI conviene revisar:

- Jobs generados por cada accion.
- Stages con mayor duracion.
- Numero de tasks por stage.
- Shuffle read y shuffle write.
- Spills a memoria o disco.

## 13. Solucion propuesta: configuracion recomendada

Para este fichero reducido de unos 20 MB, una configuracion razonable en local es:

```bash
spark-submit \
  --master local[4] \
  --conf spark.app.name=M2T4_Optimizacion_Spark \
  --conf spark.sql.shuffle.partitions=32 \
  --conf spark.sql.adaptive.enabled=true \
  --conf spark.sql.adaptive.coalescePartitions.enabled=true \
  app_sales.py \
  --input sales.csv \
  --output output/metricas_sales
```

Justificacion esperada:

- `local[4]` permite paralelismo suficiente en un portatil o equipo docente.
- `spark.sql.shuffle.partitions=32` evita el exceso de 200 particiones por defecto para un dataset pequeno.
- AQE permite que Spark ajuste particiones tras observar tamanos reales de shuffle.
- `cache()` solo debe usarse si el DataFrame limpio se reutiliza varias veces en la misma ejecucion.

## 14. Limpieza opcional

Ejecuta esta celda solo cuando hayas terminado el ejercicio y quieras cerrar la sesion Spark.

In [ ]:
# spark.stop()